In [14]:
from ModifiedNEAT.util.fancy_text import CM, Fore
from ModifiedNEAT.nn.base import Model
from ModifiedNEAT.nn.modules.sub import Linear, Conv1d, Transpose, ResidualBlock, Sequential, GroupNorm, ConverBase
from ModifiedNEAT.nn.modules import Reformer
from ModifiedNEAT.util.datetime import unix_to_datetime_file

import ModifiedNEAT as neat
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch import Tensor
from numba import njit, prange
from numba.typed import List, Dict
from typing import Union

import pygame
import random
import os
import time as clock
import numpy as np

In [15]:
DEVICE = 'cpu' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.float64

In [16]:
class RModel(Model):
    def __init__(self, inputs: int, outputs: int, seq_len: int, dim_size: int, layers: int,
                 kernel_size=1, heads: int = None, kv_heads: int = None, differential: int = False, norm_groups=1,
                 bias=False, device: torch.device = 'cpu', dtype: torch.device = torch.float32):
        super().__init__()
        # Attributes
        self.inputs         = inputs
        self.outputs        = outputs
        self.dim_size       = dim_size
        self.layers         = layers
        self.distribution   = 'normal'
        self.kernel_size    = kernel_size
        self.stride         = 1
        self.norm_groups    = norm_groups
        self.differential   = differential

        # Build
        self.pri_actv = nn.SiLU()
        self.pol_proj = neat.nn.Sequential(*[
            Transpose(),
            # GroupNorm(1, inputs, affine=True, bias=bias, device=device, dtype=dtype),
            Conv1d(inputs, dim_size, self.kernel_size, self.stride, -1, bias=bias, device=device, dtype=dtype),
            ResidualBlock(dim_size, dim_size, self.kernel_size, self.norm_groups,
                          bias, device, dtype, image_ndim=1, actv=self.pri_actv),
            ConverBase((seq_len,), dim_size, self.kernel_size, self.norm_groups, layers, heads, kv_heads,
                       self.differential, True, bias, device, dtype, actv=self.pri_actv, auto_single=True),
            ResidualBlock(dim_size, dim_size, 1, self.norm_groups,
                          bias, device, dtype, image_ndim=1, actv=self.pri_actv),
            nn.Flatten(-2, -1),
            self.pri_actv,
        ])
        self.mean_log_std = Linear(dim_size, 2*outputs, bias, device, dtype)
        self.sec_actv   = None
        self.val_proj   = neat.nn.Sequential(*[
            Transpose(),
            Conv1d(inputs, dim_size, self.kernel_size, self.stride, -1, bias=bias, device=device, dtype=dtype),
            ResidualBlock(dim_size, dim_size, self.kernel_size, self.norm_groups,
                          bias, device, dtype, image_ndim=1, actv=self.pri_actv),
            ConverBase((seq_len,), dim_size, self.kernel_size, self.norm_groups, layers, heads, kv_heads,
                       self.differential, True, bias, device, dtype, actv=self.pri_actv, auto_single=True),
            # ResidualBlock(dim_size, dim_size, 1, self.norm_groups,
            #               bias, device, dtype, image_ndim=1, actv=self.pri_actv),
            nn.Flatten(-2, -1),
            self.pri_actv,
        ])
        self.decode     = Linear(dim_size, 1, bias, device, dtype)

    def forward(self, state: Tensor, keys: Union[int, list[int]] = None, **kwargs):
        return self.get_policy(state, keys=keys, **kwargs)

    def get_mean_std(self, latent: Tensor, keys: Union[int, list[int]] = None) -> Tensor:
        mean_std        = self.mean_log_std(latent, keys=keys)
        mean, log_std   = torch.chunk(mean_std, 2, -1)
        mean            = F.sigmoid(mean)
        std             = torch.pow(10, F.sigmoid(log_std) * 3 + -4)
        return mean, std

    def get_action(self, state: Tensor, keys: Union[int, list[int]] = None) -> tuple[Tensor, Tensor]:
        latent      = self.pol_proj(state, keys=keys)
        mean, std   = self.get_mean_std(latent, keys=keys)
        dist        = torch.distributions.Normal(mean, std)
        action      = dist.sample()
        if self.sec_actv is not None:
            action = self.sec_actv(action)
        log_prob    = dist.log_prob(action)
        return action, log_prob

    def evaluate_action(self, state: Tensor, action: Tensor, keys: Union[int, list[int]] = None) -> [Tensor, Union[Tensor, None]]:
        latent      = self.pol_proj(state, keys=keys)
        mean, std   = self.get_mean_std(latent, keys=keys)
        dist        = torch.distributions.Normal(mean, std)
        log_prob    = dist.log_prob(action)
        entropy     = dist.entropy()
        return log_prob, entropy

    def get_policy(self, state: Tensor, keys: Union[int, list[int]] = None, **options) -> Tensor:
        latent      = self.pol_proj(state, keys=keys)
        mean, std   = self.get_mean_std(latent, keys=keys)
        dist        = torch.distributions.Normal(mean, std)
        action      = dist.sample()
        if self.sec_actv is not None:
            action = self.sec_actv(action)
        return action

    def get_value(self, state: Tensor, keys: Union[int, list[int]] = None) -> Tensor:
        latent      = self.val_proj(state, keys=keys)
        value       = self.decode(latent, keys=keys)
        return value

In [17]:
GENOMES     = 100
INPUTS      = 5
OUTPUTS     = 1
EMBED_SIZE  = 4
KERNEL_SIZE = 1
NORM_GROUPS = 1
SEQ_LEN     = 4
LAYERS      = 1
HEADS       = 1
KV_HEADS    = None
ENABLE_BIAS = True
DIFFERENTIAL = False
GAMMA       = np.exp(np.log(0.10) / 3)
ALPHA       = np.exp(np.log(1.5) / (2 - 1))
LOSS_REG    = 0.

In [18]:
MODEL = RModel(INPUTS, OUTPUTS, SEQ_LEN, EMBED_SIZE, LAYERS, KERNEL_SIZE, HEADS, KV_HEADS, DIFFERENTIAL, NORM_GROUPS,
               ENABLE_BIAS, DEVICE, DTYPE)
MODEL1 = RModel(INPUTS, OUTPUTS*3, SEQ_LEN, EMBED_SIZE, LAYERS, KERNEL_SIZE, HEADS, KV_HEADS, DIFFERENTIAL, NORM_GROUPS,
                ENABLE_BIAS, DEVICE, DTYPE)

In [19]:
len(MODEL.neat_parameters()), len(MODEL1.neat_parameters())

(70, 70)

In [20]:
[param.shape for param in MODEL.neat_parameters()]

[torch.Size([1, 4, 5, 1]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 4, 4, 1]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 4, 4, 1]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 4, 4, 1]),
 torch.Size([1, 4]),
 torch.Size([1, 4, 4, 1]),
 torch.Size([1, 4]),
 torch.Size([1, 4, 4, 1]),
 torch.Size([1, 4]),
 torch.Size([1, 4, 4, 1]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 4, 4, 1]),
 torch.Size([1, 4]),
 torch.Size([1, 4, 4, 1]),
 torch.Size([1, 4]),
 torch.Size([1, 4, 4, 1]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 4, 4, 1]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 4, 4, 1]),
 torch.Size([1, 4]),
 torch.Size([1, 4, 2]),
 torch.Size([1, 2]),
 torch.Size([1, 4, 5, 1]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 

In [21]:
[param.shape for param in MODEL1.neat_parameters()]

[torch.Size([1, 4, 5, 1]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 4, 4, 1]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 4, 4, 1]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 4, 4, 1]),
 torch.Size([1, 4]),
 torch.Size([1, 4, 4, 1]),
 torch.Size([1, 4]),
 torch.Size([1, 4, 4, 1]),
 torch.Size([1, 4]),
 torch.Size([1, 4, 4, 1]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 4, 4, 1]),
 torch.Size([1, 4]),
 torch.Size([1, 4, 4, 1]),
 torch.Size([1, 4]),
 torch.Size([1, 4, 4, 1]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 4, 4, 1]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 4, 4, 1]),
 torch.Size([1, 4]),
 torch.Size([1, 4, 6]),
 torch.Size([1, 6]),
 torch.Size([1, 4, 5, 1]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 4]),
 torch.Size([1, 

In [24]:
def check_param_compatibility(parameters: list[neat.NeatParameter]):
    if len(parameters) == 0:
        return False

    reference_ndim = parameters[0].ndim
    for p in parameters:
        if p.ndim != reference_ndim:
            return False

    reference_shape = parameters[0].original_shape
    for p in parameters:
        for p_dim, ref_dim in zip(p.original_shape, reference_shape):
            if p_dim != ref_dim:
                return False

    return True

In [25]:
for param_group in zip(*[m.neat_parameters() for m in [MODEL, MODEL1]]):
    print(f"{check_param_compatibility(param_group)} => {[param.shape for param in param_group]}")

True => [torch.Size([1, 4, 5, 1]), torch.Size([1, 4, 5, 1])]
True => [torch.Size([1, 4]), torch.Size([1, 4])]
True => [torch.Size([1, 4]), torch.Size([1, 4])]
True => [torch.Size([1, 4]), torch.Size([1, 4])]
True => [torch.Size([1, 4, 4, 1]), torch.Size([1, 4, 4, 1])]
True => [torch.Size([1, 4]), torch.Size([1, 4])]
True => [torch.Size([1, 4]), torch.Size([1, 4])]
True => [torch.Size([1, 4]), torch.Size([1, 4])]
True => [torch.Size([1, 4, 4, 1]), torch.Size([1, 4, 4, 1])]
True => [torch.Size([1, 4]), torch.Size([1, 4])]
True => [torch.Size([1, 4]), torch.Size([1, 4])]
True => [torch.Size([1, 4]), torch.Size([1, 4])]
True => [torch.Size([1, 4, 4, 1]), torch.Size([1, 4, 4, 1])]
True => [torch.Size([1, 4]), torch.Size([1, 4])]
True => [torch.Size([1, 4, 4, 1]), torch.Size([1, 4, 4, 1])]
True => [torch.Size([1, 4]), torch.Size([1, 4])]
True => [torch.Size([1, 4, 4, 1]), torch.Size([1, 4, 4, 1])]
True => [torch.Size([1, 4]), torch.Size([1, 4])]
True => [torch.Size([1, 4, 4, 1]), torch.Size(